# Prediction Pipeline Development

This notebook develops the prediction pipeline for the deployed permeability prediction model.

The objective is to create a workflow that accepts a SMILES string as input, generates the required molecular features, loads the saved Hybrid Random Forest model, and predicts permeability.

The workflow includes:

- Loading deployment assets
- Molecular feature generation
- Prediction generation
- Applicability Domain assessment
- User-friendly result reporting

## Import Required Libraries

The required libraries are imported for molecular feature generation, model loading, and prediction.

In [2]:
import joblib
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

## Loading Deployment Assets

The previously saved model and reference chemical space are loaded.

These assets will be used to generate predictions and perform Applicability Domain checks.

In [6]:
rf_model = joblib.load(
    "../Model_Development_and_Saving.ipynb/hybrid_rf_model.pkl"
)

reference_features = joblib.load(
    "../Model_Development_and_Saving.ipynb/training_reference_features.pkl"
)

metadata = joblib.load(
    "../Model_Development_and_Saving.ipynb/model_metadata.pkl"
)

print("Assets loaded successfully.")

Assets loaded successfully.


## Creating Hybrid Feature Generation Function

A reusable function is created to convert a SMILES string into the Hybrid molecular representation used during model training.

The generated feature vector contains:

- 2048 Morgan fingerprint bits
- 6 physicochemical descriptors

Total features: 2054

In [7]:
def smiles_to_features(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )

    fingerprint = np.array(fp)

    descriptors = np.array([

        Descriptors.MolWt(mol),

        Descriptors.MolLogP(mol),

        Descriptors.TPSA(mol),

        Descriptors.NumHDonors(mol),

        Descriptors.NumHAcceptors(mol),

        Descriptors.NumRotatableBonds(mol)

    ])

    features = np.hstack(
        [fingerprint, descriptors]
    )

    return features.reshape(1, -1)

## Testing Feature Generation

A sample molecule is converted into the Hybrid feature representation to verify that feature generation is functioning correctly.

In [9]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [10]:
test_features = smiles_to_features("CCO")

print(test_features.shape)

(1, 2054)


## Generating Permeability Prediction

The generated hybrid molecular features are passed to the saved Hybrid Random Forest model to predict permeability.

This simulates the prediction process that will later be used in the deployed application.

In [11]:
prediction = rf_model.predict(
    test_features
)

print(
    "Predicted logPapp:",
    prediction[0]
)

Predicted logPapp: 1.4749298172408367


## Creating a Reusable Prediction Function

A reusable prediction function is created to automate feature generation and permeability prediction for any valid SMILES string.

This function will later form the core of the deployed application.

In [12]:
def predict_permeability(smiles):

    features = smiles_to_features(smiles)

    if features is None:
        return "Invalid SMILES"

    prediction = rf_model.predict(features)

    return prediction[0]

In [13]:
result = predict_permeability("CCO")

print("Predicted logPapp:", result)

Predicted logPapp: 1.4749298172408372


## Importing Nearest Neighbor Library

Nearest Neighbor analysis is used to compare new molecules against the training chemical space.

This enables Applicability Domain assessment and prediction confidence estimation.

In [14]:
from sklearn.neighbors import NearestNeighbors

## Constructing the Applicability Domain Reference Model

A nearest-neighbor model is fitted using the saved training reference features.

This model represents the chemical space learned during model development and will be used to evaluate new molecules.

In [21]:
nn = NearestNeighbors(
    n_neighbors=2,
    metric='euclidean'
)

nn.fit(reference_features)

print("Applicability Domain model created.")

Applicability Domain model created.


## Loading Final Applicability Domain Threshold

The deployment-ready Applicability Domain threshold developed during final AD validation is loaded and used to assess prediction reliability.

In [22]:
ad_threshold = joblib.load(
    "../Applicability_Domain_QSAR/final_ad_threshold.pkl"
)

print("AD Threshold:", ad_threshold)

AD Threshold: 22.853484711656098


In [25]:
def check_applicability_domain(features):

    distances, _ = nn.kneighbors(features)

    # Ignore self-match and use nearest real neighbor
    distance = distances[0][0]

    if distance <= ad_threshold:

        status = "Inside AD"

        confidence = "High"

    else:

        status = "Outside AD"

        confidence = "Low"

    return distance, status, confidence

In [38]:
def check_applicability_domain(features):

    distances, _ = nn.kneighbors(features)

    # If identical molecule exists in reference space
    if distances[0][0] == 0:

        distance = distances[0][1]

    else:

        distance = distances[0][0]

    if distance <= ad_threshold:

        status = "Inside AD"
        confidence = "High"

    else:

        status = "Outside AD"
        confidence = "Low"

    return distance, status, confidence

In [39]:
print(
    predict_permeability_with_ad(
        "CCO"
    )
)

{'Predicted logPapp': 1.4749298172408367, 'Distance': 47.61072471660141, 'AD Status': 'Outside AD', 'Confidence': 'Low'}


In [40]:
print(
    predict_permeability_with_ad(
        "CC(=O)Oc1ccccc1C(=O)O"
    )
)

{'Predicted logPapp': 0.6257579680247851, 'Distance': 6.835486446479146, 'AD Status': 'Inside AD', 'Confidence': 'High'}


# Prediction Pipeline Summary

A deployment-ready prediction pipeline was developed for permeability prediction using the final Hybrid Random Forest model.

The pipeline performs:

- Molecular feature generation from SMILES
- Permeability prediction
- Applicability Domain assessment
- Confidence estimation

The resulting workflow can be integrated directly into web applications and deployed prediction services.

Example Outputs:

Ethanol:
- Outside Applicability Domain
- Low Confidence

Aspirin:
- Inside Applicability Domain
- High Confidence

These results demonstrate the ability of the pipeline to distinguish between molecules that are well represented within the training chemical space and those that fall outside the model's domain of applicability.

In [41]:
joblib.dump(
    ad_threshold,
    "final_ad_threshold.pkl"
)

print("Final AD threshold saved.")

Final AD threshold saved.
